**RETRIVAL AUGMENTED GENERATION(RAG)**

This is a technique used in Natural Language Processing that combines:

a) Retrieval: Finding relevant information from a large collection. LLMs have a knowledge cut of date and also don’t interact with private data. Collection can be a database, PDFs, or web scraping a website.

b) Generation: Using LLMs to generate answers based on the query(user input) and the retrieved data from our collection.

**RAG process**
1. Load Data- We extract text from our data sources. For this tutorial, we will use PDFs. Llama_index has a function for this called SimpleDirectoryReader.

2. Splitting Text- We then split the text into chunks, and this is also done by the SimpleDirectoryReader function when we call it. This is significant in passing it into our LLM and also searching through the information for similar chunks to the user’s query.

3. Embeddings- This refers to the numerical representation of texts. As machines understand binary. We convert our chunks into embeddings using a llama_index function called HuggingFaceEmbedding, which is open-source.

4. Vector Store- After converting texts into embeddings, we store them in a vector database such as Chroma DB or FAISS.

5. Last Step carries two tasks:

*Retrieve*- Given user input, relevant splits are retrieved from storage (Vector Database) using a retriever. In llama_index, we will use a function called as_query_engine.

*Generate*- We call our LLM to produce an answer using a prompt that includes the question and the retrieved data from our vector database.

In [6]:
#!/usr/bin/python3
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Sequence
from operator import add as add_messages
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph,END
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


In [ ]:
load_dotenv()

#set up our LLM in our casee llama on groq 
llm = ChatGroq(
    temperature=0.3,  # Lower temperature for more factual responses
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)

In [ ]:
#---Create the embedding Model
# Load embeddings with proper settings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': False}  # Chroma handles normalization
)
